# Notebook 9: Gradient Boosting

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 3.5 hours | **Week 9 — Tree-Based Methods & Gradient Boosting**

---

## Why This Matters

Gradient Boosting is the algorithm behind many Kaggle-winning solutions and real-world production models. It consistently outperforms random forests on tabular data, is the foundation of XGBoost/LightGBM/CatBoost, and teaches you a deep idea: **fitting models to errors, not data**. Understanding gradient boosting from scratch means you understand the entire family of boosting algorithms used in industry today.

---

## Real-World Analogy: The Team of Analysts

Imagine a team of analysts estimating a house price:

- **Analyst 1** looks only at square footage and gives a rough estimate: **$350K**
  - Actual price: $400K → Error = +$50K (we underestimated by $50K)
- **Analyst 2** doesn't re-estimate the house from scratch. They look at the **error** ($50K) and say: "This house is in a top school district — the error is probably $40K of that."
  - Running total: $350K + $40K = $390K → Remaining error = +$10K
- **Analyst 3** models the remaining error: "It has a recent renovation — that accounts for +$10K."
  - Final estimate: $390K + $10K = **$400K** ✓

Each analyst attacks the **residual** from the previous analyst, not the original problem. This is the essence of gradient boosting.

---

## Learning Objectives

By the end of this notebook you will be able to:
1. Explain forward stagewise additive modeling from first principles
2. Implement gradient boosting from scratch for MSE loss
3. Understand how gradient boosting generalizes to any differentiable loss
4. Tune learning rate, number of trees, depth, and subsample
5. Visualize residual shrinkage and learning dynamics

## Section 1: Theory — Forward Stagewise Additive Modeling

### The Big Idea

We want to build a strong predictor F(x) as a **sum of weak learners** (shallow trees):

$$F(x) = \sum_{m=1}^{M} \gamma_m h_m(x)$$

Instead of fitting all trees simultaneously (impossible), we add **one tree at a time**, keeping all previous trees fixed.

### Algorithm: Forward Stagewise Additive Modeling

```
Initialize: F_0(x) = mean(y)   [best constant prediction]

For m = 1 to M:
    Compute residuals: r_im = y_i - F_{m-1}(x_i)   [what's left to explain]
    Fit a tree h_m to predict r_im   [model the errors, not y!]
    Update: F_m(x) = F_{m-1}(x) + η × h_m(x)   [η = learning rate]
```

### Why Residuals = Negative Gradient of MSE

For MSE loss: $L(y, F) = \frac{1}{2}(y - F)^2$

The negative gradient with respect to F is:
$$-\frac{\partial L}{\partial F} = -\frac{\partial}{\partial F}\frac{1}{2}(y-F)^2 = (y - F)$$

This is exactly the **residual**! So fitting trees to residuals IS gradient descent — but in **function space**, not parameter space.

### The Gradient Boosting Generalization

For **any** differentiable loss $L(y, F)$, define pseudo-residuals:
$$r_{im} = -\frac{\partial L(y_i, F_{m-1}(x_i))}{\partial F_{m-1}(x_i)}$$

Then fit a tree to these pseudo-residuals. This works for:
- **MSE (regression):** pseudo-residuals = $y_i - F_{m-1}(x_i)$ (actual residuals)
- **Log-loss (classification):** pseudo-residuals = $y_i - \sigma(F_{m-1}(x_i))$
- **Huber loss:** robust to outliers
- **Quantile loss:** predicts intervals

### Learning Rate (Shrinkage)

The update rule includes a **learning rate** $\eta \in (0, 1]$:
$$F_m(x) = F_{m-1}(x) + \eta \times h_m(x)$$

| Learning Rate | Effect |
|:---:|:---|
| Large (0.5–1.0) | Big steps → fewer trees needed → higher overfitting risk |
| Small (0.01–0.1) | Small steps → more trees needed → better generalization |

**Rule of thumb:** "Shrinkage + more trees" almost always beats "large η + few trees".

### Stochastic Gradient Boosting

Add randomness by training each tree on a **random subsample** of training data (without replacement). Like SGD vs. GD — the noise from subsampling acts as regularization and speeds up training.

In [ ]:
# ── Setup & Imports ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')

print("Libraries loaded successfully.")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## Section 2: House Price Dataset

In [ ]:
# ── Generate House Price Dataset ─────────────────────────────────────────────
np.random.seed(42)
n_samples = 1000

# Feature generation
sqft        = np.random.normal(1800, 500, n_samples).clip(500, 5000)
bedrooms    = np.random.choice([2, 3, 4, 5], n_samples, p=[0.2, 0.45, 0.25, 0.10])
bathrooms   = np.random.choice([1, 2, 3, 4], n_samples, p=[0.15, 0.50, 0.25, 0.10])
age         = np.random.randint(0, 60, n_samples)
school_dist = np.random.uniform(0, 10, n_samples)   # 0=best school, 10=worst
garage      = np.random.choice([0, 1, 2], n_samples, p=[0.15, 0.40, 0.45])
renovated   = np.random.choice([0, 1], n_samples, p=[0.65, 0.35])
crime_rate  = np.random.uniform(0, 10, n_samples)

# Price formula (non-linear relationships intentional)
price = (
    120 * sqft / 1000
    + 15 * bedrooms
    + 18 * bathrooms
    - 1.2 * age
    - 8  * school_dist          # far from school = cheaper
    + 12 * garage
    + 25 * renovated
    - 5  * crime_rate
    + 0.002 * sqft * renovated  # interaction: renovation matters more in big houses
    + np.random.normal(0, 15, n_samples)  # noise
) * 1000

# Assemble DataFrame
df = pd.DataFrame({
    'sqft':        sqft,
    'bedrooms':    bedrooms,
    'bathrooms':   bathrooms,
    'age':         age,
    'school_dist': school_dist,
    'garage':      garage,
    'renovated':   renovated,
    'crime_rate':  crime_rate,
    'price':       price
})

print(f"Dataset shape: {df.shape}")
print(f"\nPrice stats:")
print(df['price'].describe().apply(lambda x: f"${x:,.0f}"))
df.head()

In [ ]:
# ── Train/Validation Split ────────────────────────────────────────────────────
feature_cols = ['sqft', 'bedrooms', 'bathrooms', 'age',
                'school_dist', 'garage', 'renovated', 'crime_rate']

X = df[feature_cols].values
y = df['price'].values

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set:   {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"\nBaseline RMSE (predict mean): ${np.sqrt(mean_squared_error(y_val, np.full_like(y_val, y_train.mean()))):,.0f}")

## Section 3: Gradient Boosting from Scratch

We implement the algorithm exactly as described: initialize with the mean, then repeatedly fit trees to the current residuals.

In [ ]:
# ── GradientBoostingScratch Implementation ────────────────────────────────────
class GradientBoostingScratch:
    """
    Gradient Boosting for regression (MSE loss).

    Forward Stagewise Additive Modeling:
      F_0(x) = mean(y)
      For m = 1..M:
        r_i = y_i - F_{m-1}(x_i)     [pseudo-residuals = negative gradient of MSE]
        h_m = DecisionTree.fit(X, r)  [fit tree to residuals]
        F_m = F_{m-1} + eta * h_m     [update with shrinkage]
    """
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators   = n_estimators
        self.learning_rate  = learning_rate
        self.max_depth      = max_depth
        self.trees          = []
        self.F0             = None

    def fit(self, X, y):
        # Step 0: Initialize prediction as the mean of y
        self.F0 = np.mean(y)
        Fm = np.full(len(y), self.F0)     # shape: (n_samples,)

        for m in range(self.n_estimators):
            # Step 1: Compute pseudo-residuals (negative gradient of MSE)
            residuals = y - Fm            # r_im = y_i - F_{m-1}(x_i)

            # Step 2: Fit a shallow tree to the residuals (NOT to y!)
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42)
            tree.fit(X, residuals)

            # Step 3: Update predictions with shrinkage
            Fm += self.learning_rate * tree.predict(X)

            self.trees.append(tree)
        return self

    def predict(self, X):
        """Sum of all tree predictions, starting from the mean baseline."""
        Fm = np.full(len(X), self.F0)
        for tree in self.trees:
            Fm += self.learning_rate * tree.predict(X)
        return Fm

    def staged_predict(self, X):
        """Yield predictions after each tree is added (for learning curves)."""
        Fm = np.full(len(X), self.F0)
        yield Fm.copy()                   # Stage 0: baseline (mean)
        for tree in self.trees:
            Fm += self.learning_rate * tree.predict(X)
            yield Fm.copy()               # Stage m: after m trees


# Quick test
gb_scratch = GradientBoostingScratch(n_estimators=100, learning_rate=0.1, max_depth=3)
gb_scratch.fit(X_train, y_train)

train_rmse = np.sqrt(mean_squared_error(y_train, gb_scratch.predict(X_train)))
val_rmse   = np.sqrt(mean_squared_error(y_val,   gb_scratch.predict(X_val)))
print(f"Scratch GB — Train RMSE: ${train_rmse:,.0f}  |  Val RMSE: ${val_rmse:,.0f}")

## Section 4: Residual Shrinkage Visualization

We visualize how residuals shrink as more trees are added. Each panel shows residuals vs. predicted values at a different boosting stage.

In [ ]:
# ── Residual Shrinkage: Scatter at Stages 1, 5, 20, 100 ──────────────────────
stages_to_plot = [1, 5, 20, 100]

# Collect staged predictions on training data
staged_preds = list(gb_scratch.staged_predict(X_train))
# staged_preds[0] = baseline (mean), staged_preds[m] = after m trees

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Residual Shrinkage: How Gradient Boosting Attacks Errors Stage by Stage',
             fontsize=13, fontweight='bold', y=1.02)

for ax, stage in zip(axes, stages_to_plot):
    preds     = staged_preds[stage]          # predictions after `stage` trees
    residuals = y_train - preds
    rmse      = np.sqrt(np.mean(residuals**2))

    ax.scatter(preds / 1000, residuals / 1000,
               alpha=0.3, s=12, color='steelblue', edgecolors='none')
    ax.axhline(0, color='crimson', lw=1.5, linestyle='--', label='Zero residual')
    ax.set_xlabel('Predicted Price ($K)', fontsize=10)
    ax.set_ylabel('Residual ($K)', fontsize=10)
    ax.set_title(f'After {stage} Tree{"s" if stage>1 else ""}\nRMSE = ${rmse:,.0f}', fontsize=10)
    ax.set_ylim(-200, 200)

    # Annotate residual spread
    std_res = np.std(residuals) / 1000
    ax.text(0.05, 0.93, f'Std: ±${std_res:.1f}K',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()
print("\nKey insight: The residuals compress toward zero as more trees are added.")
print("Early trees handle large errors; later trees refine small remaining errors.")

## Section 5: Verify Against Sklearn's GradientBoostingRegressor

In [ ]:
# ── Verify Scratch vs Sklearn ─────────────────────────────────────────────────
gb_sklearn = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=3,
    subsample=1.0, random_state=42
)
gb_sklearn.fit(X_train, y_train)

pred_scratch = gb_scratch.predict(X_val)
pred_sklearn = gb_sklearn.predict(X_val)

rmse_scratch = np.sqrt(mean_squared_error(y_val, pred_scratch))
rmse_sklearn = np.sqrt(mean_squared_error(y_val, pred_sklearn))
max_diff     = np.max(np.abs(pred_scratch - pred_sklearn))

print("=" * 52)
print(f"{'Model':<30} {'Val RMSE':>10}")
print("-" * 52)
print(f"{'GradientBoostingScratch':<30} ${rmse_scratch:>9,.0f}")
print(f"{'sklearn GradientBoosting':<30} ${rmse_sklearn:>9,.0f}")
print("=" * 52)
print(f"\nMax absolute difference in predictions: ${max_diff:,.0f}")
print("(Small differences due to leaf optimization details in sklearn)")

# Visual comparison: scatter plot of predictions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, preds, label, color in [
    (axes[0], pred_scratch, 'Scratch GB',  'steelblue'),
    (axes[1], pred_sklearn, 'Sklearn GB', 'darkorange')
]:
    ax.scatter(y_val / 1000, preds / 1000, alpha=0.4, s=15, color=color)
    mn = min(y_val.min(), preds.min()) / 1000
    mx = max(y_val.max(), preds.max()) / 1000
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1.5, label='Perfect prediction')
    ax.set_xlabel('Actual Price ($K)')
    ax.set_ylabel('Predicted Price ($K)')
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    ax.set_title(f'{label}\nRMSE = ${rmse:,.0f}')
    ax.legend(fontsize=8)

plt.suptitle('Scratch vs. Sklearn: Actual vs. Predicted', fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6: Learning Rate Effect

Different learning rates require different numbers of trees. A small learning rate needs many trees but generalizes better. We compare staged RMSE curves for η = 0.001, 0.01, 0.1, 0.5.

In [ ]:
# ── Learning Rate Effect: Staged Train/Val RMSE Curves ──────────────────────
# Different learning rates need different n_estimators to converge
lr_configs = [
    {'lr': 0.001, 'n_est': 3000, 'color': 'purple',     'label': 'η=0.001 (very slow)'},
    {'lr': 0.01,  'n_est': 1000, 'color': 'steelblue',  'label': 'η=0.01  (slow)'},
    {'lr': 0.1,   'n_est': 300,  'color': 'forestgreen', 'label': 'η=0.1   (balanced)'},
    {'lr': 0.5,   'n_est': 100,  'color': 'crimson',    'label': 'η=0.5   (fast/risky)'},
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cfg in lr_configs:
    gb = GradientBoostingRegressor(
        n_estimators=cfg['n_est'], learning_rate=cfg['lr'],
        max_depth=3, subsample=1.0, random_state=42
    )
    gb.fit(X_train, y_train)

    train_rmses, val_rmses = [], []
    for preds_tr, preds_vl in zip(
        gb.staged_predict(X_train), gb.staged_predict(X_val)
    ):
        train_rmses.append(np.sqrt(mean_squared_error(y_train, preds_tr)))
        val_rmses.append(np.sqrt(mean_squared_error(y_val,   preds_vl)))

    stages = np.arange(len(train_rmses))

    axes[0].plot(stages, np.array(train_rmses)/1000, color=cfg['color'],
                 lw=1.5, label=cfg['label'])
    axes[1].plot(stages, np.array(val_rmses)/1000,   color=cfg['color'],
                 lw=1.5, label=cfg['label'])

    # Mark best val stage
    best_stage = np.argmin(val_rmses)
    axes[1].scatter(best_stage, val_rmses[best_stage]/1000,
                    color=cfg['color'], s=80, zorder=5, marker='*')

for ax, title in [(axes[0], 'Training RMSE'), (axes[1], 'Validation RMSE')]:
    ax.set_xlabel('Number of Trees (Boosting Rounds)')
    ax.set_ylabel('RMSE ($K)')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 120)

axes[1].set_title('Validation RMSE (★ = best stage)', fontweight='bold')
plt.suptitle('Learning Rate Effect: η vs. Number of Trees Trade-off',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey takeaway:")
print("  η=0.001 needs ~3000 trees to converge but may achieve best generalization.")
print("  η=0.5 converges fast but risks overfitting (train RMSE << val RMSE).")
print("  The ★ markers on validation curves show optimal stopping points.")

## Section 7: Depth × Learning Rate Interaction

Tree depth controls model complexity per tree. Combined with learning rate, they form an important interaction: deeper trees with small learning rate vs. shallow trees with larger learning rate.

In [ ]:
# ── Depth × Learning Rate Grid ───────────────────────────────────────────────
depths = [1, 3, 5]
lrs    = [0.01, 0.1]
n_est  = 500

results = []
for depth in depths:
    for lr in lrs:
        gb = GradientBoostingRegressor(
            n_estimators=n_est, learning_rate=lr,
            max_depth=depth, subsample=1.0, random_state=42
        )
        gb.fit(X_train, y_train)

        # Find best val RMSE via staged prediction
        val_rmses = [
            np.sqrt(mean_squared_error(y_val, p))
            for p in gb.staged_predict(X_val)
        ]
        best_rmse  = min(val_rmses)
        best_stage = np.argmin(val_rmses)
        train_rmse = np.sqrt(mean_squared_error(y_train, gb.predict(X_train)))

        results.append({
            'max_depth':       depth,
            'learning_rate':   lr,
            'best_val_RMSE':   f'${best_rmse:,.0f}',
            'best_n_trees':    best_stage,
            'train_RMSE':      f'${train_rmse:,.0f}',
        })

results_df = pd.DataFrame(results)
print("Depth × Learning Rate — Validation RMSE Grid")
print("=" * 65)
print(results_df.to_string(index=False))
print()

# Heatmap of numeric RMSE
pivot_data = []
for depth in depths:
    row = []
    for lr in lrs:
        gb = GradientBoostingRegressor(
            n_estimators=n_est, learning_rate=lr,
            max_depth=depth, random_state=42
        )
        gb.fit(X_train, y_train)
        val_rmse = np.sqrt(mean_squared_error(y_val, gb.predict(X_val)))
        row.append(val_rmse / 1000)
    pivot_data.append(row)

pivot_df = pd.DataFrame(
    pivot_data,
    index=[f'depth={d}' for d in depths],
    columns=[f'η={l}' for l in lrs]
)

fig, ax = plt.subplots(figsize=(5, 3.5))
sns.heatmap(pivot_df, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Val RMSE ($K)'})
ax.set_title('Validation RMSE ($K) — Depth × Learning Rate', fontweight='bold')
plt.tight_layout()
plt.show()

## Section 8: Subsample Effect (Stochastic Gradient Boosting)

In [ ]:
# ── Subsample Effect: Compare 0.5, 0.8, 1.0 ──────────────────────────────────
subsample_vals = [0.5, 0.8, 1.0]
colors         = ['tomato', 'steelblue', 'forestgreen']
n_est          = 300

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for sub, color in zip(subsample_vals, colors):
    gb = GradientBoostingRegressor(
        n_estimators=n_est, learning_rate=0.05,
        max_depth=3, subsample=sub, random_state=42
    )
    gb.fit(X_train, y_train)

    train_rmses = [np.sqrt(mean_squared_error(y_train, p))
                   for p in gb.staged_predict(X_train)]
    val_rmses   = [np.sqrt(mean_squared_error(y_val, p))
                   for p in gb.staged_predict(X_val)]

    stages = np.arange(len(train_rmses))
    label  = f'subsample={sub}'
    axes[0].plot(stages, np.array(train_rmses)/1000, color=color,
                 lw=1.5, label=label)
    axes[1].plot(stages, np.array(val_rmses)/1000,   color=color,
                 lw=1.5, label=label)

    best = np.argmin(val_rmses)
    axes[1].scatter(best, val_rmses[best]/1000, color=color, s=80, zorder=5)

for ax, title in [(axes[0], 'Train RMSE ($K)'), (axes[1], 'Validation RMSE ($K)')]:
    ax.set_xlabel('Number of Trees')
    ax.set_ylabel('RMSE ($K)')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Stochastic Gradient Boosting: Effect of Row Subsampling',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  subsample=1.0: Standard GB — training converges smoothly but may overfit.")
print("  subsample=0.8: Slight noise in train curve, often better validation RMSE.")
print("  subsample=0.5: More noise, but strong regularization — best generalization.")
print("  The dot (•) marks the optimal number of trees for each setting.")

## Section 9: Gradient Boosting as Gradient Descent in Function Space

This 1D illustration shows how gradient boosting progressively refines the prediction function F(x), converging toward the true function step by step — just like gradient descent converges toward the minimum of a loss.

In [ ]:
# ── 1D Illustration: F(x) Converging to Truth ────────────────────────────────
np.random.seed(42)

# 1D data: non-linear house price vs. sqft
x_1d   = np.linspace(500, 4000, 200)
y_true = 80 + 0.05 * x_1d + 30 * np.sin(x_1d / 400) + 15 * np.cos(x_1d / 800)

# Add noise for training data
x_train_1d = np.random.uniform(500, 4000, 150)
y_train_1d = 80 + 0.05 * x_train_1d + 30 * np.sin(x_train_1d / 400) + \
             15 * np.cos(x_train_1d / 800) + np.random.normal(0, 8, 150)

X_1d_train = x_train_1d.reshape(-1, 1)
X_1d_grid  = x_1d.reshape(-1, 1)

stages_show = [1, 3, 10, 50]
gb_1d = GradientBoostingRegressor(
    n_estimators=max(stages_show), learning_rate=0.3,
    max_depth=2, random_state=42
)
gb_1d.fit(X_1d_train, y_train_1d)
staged_preds_1d = list(gb_1d.staged_predict(X_1d_grid))

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
fig.suptitle('Gradient Boosting in Function Space: F(x) Converging to Truth',
             fontsize=12, fontweight='bold')

for ax, stage in zip(axes, stages_show):
    ax.scatter(x_train_1d, y_train_1d, alpha=0.3, s=10, color='gray', label='Data')
    ax.plot(x_1d, y_true, 'k--', lw=2, label='True f(x)')
    ax.plot(x_1d, staged_preds_1d[stage], color='crimson', lw=2,
            label=f'F_{stage}(x)')

    # Shade the gap between prediction and truth
    ax.fill_between(x_1d, staged_preds_1d[stage], y_true,
                    alpha=0.15, color='red', label='Residual')

    residual_rmse = np.sqrt(np.mean((y_true - staged_preds_1d[stage])**2))
    ax.set_title(f'm = {stage} trees\nRMSE={residual_rmse:.1f}', fontsize=10)
    ax.set_xlabel('Sqft')
    if ax == axes[0]:
        ax.set_ylabel('Price ($K)')
    ax.legend(fontsize=7, loc='upper left')

plt.tight_layout()
plt.show()

print("\nAt m=1: F(x) is a single tree — rough step function.")
print("At m=3: Some curvature captured, large residuals remain (red shading).")
print("At m=10: Shape closer to truth, residuals shrinking.")
print("At m=50: F(x) closely tracks the true function.")
print("\nEach step: the new tree models the RESIDUAL (gap), pulling F(x) toward truth.")

## Section 10: Summary & Key Hyperparameters

| Hyperparameter | Typical Range | Effect |
|:---|:---:|:---|
| `n_estimators` | 100–3000 | More trees = better (up to a point) — use early stopping |
| `learning_rate` | 0.01–0.3 | Smaller = more robust, needs more trees |
| `max_depth` | 1–5 | Depth 1 = stumps (additive model); depth 3–5 captures interactions |
| `subsample` | 0.5–1.0 | <1.0 adds stochasticity → regularizes + speeds up |
| `min_samples_leaf` | 1–50 | Prevents tiny, overfit leaves |

### The Bias-Variance Trade-off in Boosting

- **Bias** reduced by: more trees, larger learning rate, deeper trees
- **Variance** reduced by: smaller learning rate, subsample < 1, shallower trees

### Gradient Boosting vs. Random Forest

| Property | Random Forest | Gradient Boosting |
|:---|:---:|:---:|
| Base learners | Independent | Sequential (each fixes previous) |
| Parallelism | Easy | Harder (sequential by design) |
| Tuning effort | Low | Higher (learning rate + n_trees) |
| Typical performance | Good | Better (on tabular data) |
| Overfitting risk | Lower | Higher (but manageable) |

## Self-Check Questions

Test your understanding. Try to answer before revealing the solution.

---

**Question 1:** Why does gradient boosting use `learning_rate < 1`? Mathematically, setting η=1 means taking a full gradient step — why is this worse?

<details>
<summary>Click to reveal answer</summary>

**Answer:**

With η=1, each tree fully corrects the current residuals on the **training data**. But the tree was fit to training residuals specifically — it has overfit to the training noise. When you take a full step (η=1), you propagate that overfitting directly into F(x).

Mathematically, this is analogous to gradient descent with a large learning rate: you can overshoot the minimum and oscillate or diverge. In function space, η=1 means F(x) memorizes training errors rather than learning the underlying pattern.

With η < 1 (shrinkage), each tree contributes only a fraction of its correction. This **dampens overfitting** and forces the model to use more trees — but each tree's contribution is "averaged" with what came before, yielding a smoother, more generalizable function.

Think of it as: η=1 is a hyperactive analyst who completely revises the estimate based on one data point; η=0.1 is a careful analyst who slightly adjusts the estimate, requiring consensus across many analysts.

</details>

---

**Question 2:** Gradient boosting with MSE loss fits trees to residuals $(y - F)$. Gradient boosting with log-loss fits trees to $(y - \sigma(F))$. What is $(y - \sigma(F))$ called in logistic regression?

<details>
<summary>Click to reveal answer</summary>

**Answer:**

$(y - \sigma(F))$ is the **residual in logistic regression**, sometimes called the **Pearson residual** or simply the **response residual**.

Here, $\sigma(F) = \frac{1}{1 + e^{-F}}$ is the predicted probability. The quantity $y - \sigma(F)$ measures the difference between the true label (0 or 1) and the predicted probability.

This is also the **negative gradient of the log-loss** (binary cross-entropy):
$$-\frac{\partial}{\partial F}[-y\log(\sigma(F)) - (1-y)\log(1-\sigma(F))] = \sigma(F) - y$$

So the pseudo-residuals for classification gradient boosting are exactly the "errors" in predicted probability — the same quantity that drives parameter updates in logistic regression via gradient descent.

</details>

---

**Question 3:** You set `n_estimators=1000` and `learning_rate=0.001`. Your colleague sets `n_estimators=100` and `learning_rate=0.01`. Which do you expect to perform better and why?

<details>
<summary>Click to reveal answer</summary>

**Answer:**

Both configurations have the same **effective total step size**: $1000 \times 0.001 = 100 \times 0.01 = 1.0$. However, the key question is whether each has **converged**.

In practice:
- If neither has converged (both are underfitting), **n_estimators=1000 with η=0.001** will likely perform slightly better because smaller learning rates with more trees explore the function space more carefully and tend to generalize better — this is the "shrinkage + more trees" principle.
- If both have converged (i.e., there are enough trees to fit the training data well), they may perform similarly.

The real insight: **total accumulated step = η × M** matters for whether you've converged, but **η alone** (independent of M) affects generalization. Smaller η creates a smoother, more regularized path through function space — even with the same total accumulated step.

In most experiments: given equal computation budget, smaller learning rates with proportionally more trees win. The n_estimators=1000/η=0.001 config is preferable IF training time allows it.

</details>